# M-κ with Explicit Cracking Point — Design / Characteristic / Mean

Demonstrates the refined M-κ discretization that pins the cracking curvature
κ_cr explicitly in the sample array, eliminating solver failures at the cracking
discontinuity and ensuring the origin (0, 0) appears in the result.

**Cross-section:** T-beam from notebook 10  
b_f = 1150 mm, h_f = 150 mm, b_w = 300 mm, h_w = 250 mm → h = 400 mm  
C35/45, B500B reinforcement  
A_s,bot = 804 mm² at z = 40 mm, A_s,top = 1010 mm² at z = 360 mm

**Three resistance modes — all use EC2 parabola-rectangle:**
- **Design** — α_cc = 0.85, γ_c = 1.5, γ_s = 1.15  →  f_cd = 19.8 MPa, f_yd = 435 MPa
- **Characteristic** — α_cc = 1.0, γ_c = 1.0, γ_s = 1.0  →  f_cd = f_ck = 35 MPa, f_yd = 500 MPa
- **Mean** — f_ck_input = f_cm = f_ck + 8, α_cc = 1.0, γ_c = 1.0, γ_s = 1.0; steel f_yR = 1.1·f_yk (EC2 §5.7 NA.10)  →  f_cd = f_cm = 43 MPa, f_yd = 550 MPa

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scite.cs_design.shapes import TShape
from scite.cs_design.cross_section import CrossSection
from scite.cs_design.reinforcement import ReinforcementLayer, ReinforcementLayout
from scite.matmod.ec2_parabola_rectangle import EC2ParabolaRectangle
from scite.matmod.steel_reinforcement import SteelReinforcement
from scite.mkappa import MKappaAnalysis, MKappaModes
from scite.beam.bending.beam_deflection import BeamDeflectionAnalysis

## Cross-section parameters (same as notebook 10)

In [ ]:
# Geometry
b_f   = 1150.0   # flange width [mm]
h_f   =  150.0   # flange height [mm]
b_w   =  300.0   # web width [mm]
h_w   =  250.0   # web height [mm]
h_ref = h_w + h_f   # total height = 400 mm

# Reinforcement
A_s_bot = 804.0   # bottom layer [mm²]
A_s_top = 1010.0  # top layer [mm²]
z_s_bot = 40.0            # distance from bottom [mm]
z_s_top = h_ref - z_s_bot  # = 360 mm

# Material grades
f_ck = 35.0   # [MPa]
f_yk = 500.0  # [MPa]

shape = TShape(b_f=b_f, h_f=h_f, b_w=b_w, h_w=h_w)

print(f'T-section: {b_f:.0f}×{h_ref:.0f} mm  (web {b_w:.0f}×{h_w:.0f}, flange {b_f:.0f}×{h_f:.0f})')
print(f'A_s,bot = {A_s_bot:.0f} mm²  at z = {z_s_bot:.0f} mm')
print(f'A_s,top = {A_s_top:.0f} mm²  at z = {z_s_top:.0f} mm')

## Build three MKappaAnalysis objects

In [ ]:
def build_cs(concrete, steel):
    return CrossSection(
        shape=shape,
        concrete=concrete,
        reinforcement=ReinforcementLayout(layers=[
            ReinforcementLayer(material=steel, A_s=A_s_bot, z=z_s_bot),
            ReinforcementLayer(material=steel, A_s=A_s_top, z=z_s_top),
        ]),
    )

f_cm = f_ck + 8.0   # mean compressive strength [MPa]

# ── Design ──────────────────────────────────────────────────────────────────
concrete_d  = EC2ParabolaRectangle(f_ck=f_ck, alpha_cc=0.85, gamma_c=1.5)
steel_d     = SteelReinforcement(f_yk=f_yk, gamma_s=1.15)
mk_design   = MKappaAnalysis(cs=build_cs(concrete_d, steel_d), n_kappa=100, solve_mode='both')

# ── Characteristic ───────────────────────────────────────────────────────────
concrete_ch = EC2ParabolaRectangle(f_ck=f_ck, alpha_cc=1.0, gamma_c=1.0)
steel_ch    = SteelReinforcement(f_yk=f_yk, gamma_s=1.0)
mk_char     = MKappaAnalysis(cs=build_cs(concrete_ch, steel_ch), n_kappa=100, solve_mode='both')

# ── Mean — EC2ParabolaRectangle at f_cm, steel per EC2 §5.7 NA.10 ───────────
# Pass f_ck=f_cm so that f_cd = α_cc·f_cm/γ_c = 1.0·f_cm/1.0 = f_cm
k_ductility = SteelReinforcement(f_yk=f_yk, gamma_s=1.15).f_tk / f_yk
f_yk_mean   = 1.1 * f_yk        # = 550 MPa
f_tk_mean   = k_ductility * f_yk_mean
concrete_m  = EC2ParabolaRectangle(f_ck=f_cm, alpha_cc=1.0, gamma_c=1.0)
steel_m     = SteelReinforcement(f_yk=f_yk_mean, f_tk=f_tk_mean, gamma_s=1.0)
mk_mean     = MKappaAnalysis(cs=build_cs(concrete_m, steel_m), n_kappa=100, solve_mode='both')

modes = [
    (mk_design,  MKappaModes.COLORS['design'],         'design'),
    (mk_char,    MKappaModes.COLORS['characteristic'], 'characteristic'),
    (mk_mean,    MKappaModes.COLORS['mean'],           'mean'),
]

for mk, color, name in modes:
    mk.solve()

print(f'f_cm = {f_cm:.0f} MPa  (f_ck + 8 = {f_ck:.0f} + 8)')
print(f'  design:         f_cd = {concrete_d.f_cd:.2f} MPa,  f_yd = {steel_d.f_yd:.1f} MPa')
print(f'  characteristic: f_cd = {concrete_ch.f_cd:.2f} MPa,  f_yd = {steel_ch.f_yd:.1f} MPa')
print(f'  mean:           f_cd = {concrete_m.f_cd:.2f} MPa,  f_yd = {steel_m.f_yd:.1f} MPa  (f_yk_mean={f_yk_mean:.0f})')

## Cracking curvatures summary

In [ ]:
print(f'{"Mode":<16} {"f_ctm":>8} {"E_cm":>8} {"κ_cr,pos [‰/m]":>16} {"M_cr,pos [kN·m]":>17} {"κ_cr,neg [‰/m]":>16} {"M_cr,neg [kN·m]":>17}')
print('-' * 104)
for mk, _, name in modes:
    conc = mk.cs.concrete
    kcr_p  = mk.kappa_cr_pos * 1e6 if mk.kappa_cr_pos else float('nan')
    Mcr_p  = mk.M_cr_pos           if mk.M_cr_pos  is not None else float('nan')
    kcr_n  = mk.kappa_cr_neg * 1e6 if mk.kappa_cr_neg else float('nan')
    Mcr_n  = mk.M_cr_neg           if mk.M_cr_neg  is not None else float('nan')
    f_ctm  = getattr(conc, 'f_ctm', float('nan'))
    E_cm   = getattr(conc, 'E_cm', float('nan'))
    print(f'{name:<16} {f_ctm:>8.3f} {E_cm:>8.0f} {kcr_p:>16.4f} {Mcr_p:>17.2f} {kcr_n:>16.4f} {Mcr_n:>17.2f}')
print()
print('κ_cr in ×10⁻⁶/mm = ‰/m')
print()
print('Note: design and characteristic share the same f_ck → identical κ_cr and M_cr.')
print('      Mean mode uses f_cm = f_ck + 8, so f_ctm and E_cm are slightly higher,')
print('      giving a modestly larger κ_cr and M_cr.')
print('      M_cr = E_c·I_g·κ_cr (elastic uncracked section, model-agnostic formula).')

## M(κ) — design / characteristic / mean overlay

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))

mk_design.plot_mk(
    ax=ax,
    color=MKappaModes.COLORS['design'],
    label='M-κ  (design)',
    show_full_range=True,
    show_failure_marker=True,
    show_cracking_marker=True,
)

ax.set_xlabel('Curvature  κ  [‰/mm]', fontsize=11)
ax.set_ylabel('Moment  M  [kN·m]', fontsize=11)
ax.set_title(
    'M-κ — design mode  (T-section C35/45, B500B)\n'
    '○ = cracking point (κ_cr, M_cr)  ◆/×/○ = failure mode',
    fontsize=11, fontweight='bold',
)
ax.axhline(0, color='k', lw=0.5)
ax.axvline(0, color='k', lw=0.5)
MKappaAnalysis.add_failure_mode_legend(ax, fontsize=9, loc='lower right')
plt.tight_layout()
plt.show()

print(f'κ_cr,pos = {mk_design.kappa_cr_pos*1e6:.4f} ×10⁻⁶/mm  (= {mk_design.kappa_cr_pos*1e3:.6f} ‰/mm)')
print(f'M_cr,pos = {mk_design.M_cr_pos:.2f} kN·m   [analytical: E_c·I_g·κ_cr]')
print(f'κ_cr,neg = {mk_design.kappa_cr_neg*1e6:.4f} ×10⁻⁶/mm')
print(f'M_cr,neg = {mk_design.M_cr_neg:.2f} kN·m')
print(f'κ_max    = {mk_design.get_kappa_range()[1]*1e3:.4f} ‰/mm')
print(f'κ_cr,pos / κ_max = {mk_design.kappa_cr_pos / mk_design.get_kappa_range()[1] * 100:.2f} %  → near y-axis on full-range plot')

## κ(M) — inverted M-κ, ascending envelope

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))

for mk, color, name in modes:
    mk.plot_kappa_from_M(
        ax=ax,
        color=color,
        label_suffix=f'  ({name})',
    )

ax.set_title(
    'Inverted M-κ: κ(M) — design / characteristic / mean  (T-section C35/45, B500B)\n'
    'Solid = positive-κ (M > 0),  Dashed = negative-κ (M < 0)',
    fontsize=11, fontweight='bold',
)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Discretization detail — pre-cracking zone (zoom)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for mk, color, name in modes:
    kappa_mm = mk.kappa_values * 1e3   # → ‰/mm
    axes[0].plot(kappa_mm, mk.M_values, '-o', color=color, ms=3, lw=1.5,
                 label=name)
    axes[1].plot(kappa_mm, mk.M_values, '-o', color=color, ms=3, lw=1.5,
                 label=name)

# Left: full positive-κ range
axes[0].set_title('Positive-κ M-κ — full range', fontsize=10, fontweight='bold')
axes[0].set_xlabel('κ  [‰/mm]'); axes[0].set_ylabel('M  [kN·m]')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, ls='--')

# Right: zoom into cracking zone
kappa_zoom = max(mk.kappa_cr_pos for mk, _, _ in modes if mk.kappa_cr_pos) * 1e3 * 30
axes[1].set_xlim(-0.02e-3, kappa_zoom)
axes[1].set_title(
    f'Positive-κ M-κ — cracking zone  (κ ≤ {kappa_zoom*1e3:.1f} ×10⁻³ ‰/mm)',
    fontsize=10, fontweight='bold')
axes[1].set_xlabel('κ  [‰/mm]'); axes[1].set_ylabel('M  [kN·m]')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, ls='--')

# Mark cracking curvatures
for mk, color, name in modes:
    if mk.kappa_cr_pos:
        M_cr = np.interp(mk.kappa_cr_pos, mk.kappa_values, mk.M_values)
        for ax_ in axes:
            ax_.axvline(mk.kappa_cr_pos * 1e3, color=color, lw=0.8, ls=':', alpha=0.5)
            ax_.plot(mk.kappa_cr_pos * 1e3, M_cr, 'o', color=color, ms=6,
                     markerfacecolor='white', markeredgewidth=1.5, zorder=5)

plt.tight_layout()
plt.show()

## Peak moments summary

In [ ]:
print(f'{"Mode":<16} {"M_peak,pos [kN·m]":>19} {"M_peak,neg [kN·m]":>19} {"n_pos pts":>10} {"n_neg pts":>10}')
print('-' * 80)
for mk, _, name in modes:
    m_pos = max(mk.M_values)
    m_neg = min(mk.M_neg_values) if len(mk.M_neg_values) > 0 else float('nan')
    print(f'{name:<16} {m_pos:>19.1f} {m_neg:>19.1f} {len(mk.kappa_values):>10} {len(mk.kappa_neg_values):>10}')

## Load-deflection curves — simply supported beam, L = 6 m, distributed load

`BeamDeflectionAnalysis` integrates the M-κ relationship along the span to produce
the nonlinear q–w_max curve.  The same T-section cross-sections are reused.

- SLS limit: w = L / 250 = 24 mm (shown as dashed vertical line)
- Ultimate q_R = 8 M_R / L²  (simply supported beam, uniform load)

In [ ]:
L_beam = 6000.0   # span [mm]

bda_d    = BeamDeflectionAnalysis(cs=build_cs(concrete_d,  steel_d),
                                  L=L_beam, load_type='dist',
                                  n_kappa=100, n_load_steps=61, solve_mode='positive')
bda_ch   = BeamDeflectionAnalysis(cs=build_cs(concrete_ch, steel_ch),
                                  L=L_beam, load_type='dist',
                                  n_kappa=100, n_load_steps=61, solve_mode='positive')
bda_m    = BeamDeflectionAnalysis(cs=build_cs(concrete_m,  steel_m),
                                  L=L_beam, load_type='dist',
                                  n_kappa=100, n_load_steps=61, solve_mode='positive')

beam_modes = [
    (bda_d,  MKappaModes.COLORS['design'],         'design'),
    (bda_ch, MKappaModes.COLORS['characteristic'], 'characteristic'),
    (bda_m,  MKappaModes.COLORS['mean'],           'mean'),
]

fig, ax = plt.subplots(figsize=(9, 5.5))
for bda, color, name in beam_modes:
    q_R_kNm = bda.F_R   # [N/mm] = kN/m (BeamDeflectionAnalysis._F_scale = 1.0 for 'dist')
    bda.plot_Fw(ax=ax, color=color,
                label=f'q–w  ({name},  q_R = {q_R_kNm:.1f} kN/m)',
                show_sls=(name == 'design'))   # show SLS line once

ax.set_title(
    f'Load-deflection q–w   (T-section C35/45, B500B,  L = {L_beam/1000:.0f} m, dist. load)\n'
    'SLS limit  w = L/250 shown for design mode',
    fontsize=11, fontweight='bold',
)
ax.set_xlabel(r'$w_\mathrm{max}$  [mm]', fontsize=11)
ax.set_ylabel(r'$q$  [kN/m]', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'{"Mode":<16} {"q_R [kN/m]":>12} {"M_R [kN·m]":>13} {"w_SLS_limit [mm]":>18}')
print('-' * 65)
for bda, _, name in beam_modes:
    print(f'{name:<16} {bda.F_R:>12.1f} {bda.M_R:>13.1f} {bda.L/250:>18.1f}')